In [ ]:
import os
import json

# Replace these with your actual Kaggle username + key
kaggle_username = "shanaeo"
kaggle_key = "KGAT_d3d00c04ea7deaef3c988a0132bd4fd8"

# Create kaggle.json file
kaggle_dir = "/root/.kaggle"
os.makedirs(kaggle_dir, exist_ok=True)

with open(f"{kaggle_dir}/kaggle.json", "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)

# Set permissions
os.chmod(f"{kaggle_dir}/kaggle.json", 600)

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store
!unzip -q ecommerce-behavior-data-from-multi-category-store.zip

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors
100% 4.29G/4.29G [00:22<00:00, 201MB/s]



In [ ]:
import pandas as pd
import numpy as np

# ── 1. Load ────────────────────────────────────────────────────────────────────
df = pd.read_csv("2019-Oct.csv")

print(f"Raw shape: {df.shape}")
print(df.dtypes)
print(df.head(3))

# ── 2. Parse & standardise event_time ─────────────────────────────────────────
df["event_time"] = pd.to_datetime(df["event_time"], utc=True)

# ── 3. Select & rename to canonical columns ───────────────────────────────────
cols_needed = [
    "user_id", "event_time", "event_type",
    "product_id", "category_id", "category_code",
    "brand", "price", "user_session",
]
df = df[cols_needed].copy()

# ── 4. Fill missing categoricals ──────────────────────────────────────────────
df["brand"]         = df["brand"].fillna("unknown")
df["category_code"] = df["category_code"].fillna("unknown")
df["category_id"]   = df["category_id"].fillna(0).astype("int64")

# ── 5. Split hierarchical category_code  (e.g. "electronics.audio.headphone") ─
cat_split = df["category_code"].str.split(".", expand=True)
cat_split.columns = [f"category_level{i+1}" for i in range(cat_split.shape[1])]

# Pad to at least 3 levels
for lvl in ["category_level1", "category_level2", "category_level3"]:
    if lvl not in cat_split.columns:
        cat_split[lvl] = np.nan

cat_split = cat_split[["category_level1", "category_level2", "category_level3"]]
cat_split = cat_split.fillna("unknown")

df = pd.concat([df, cat_split], axis=1)

# ── 6. Merge very rare categories into "other" ────────────────────────────────
#    Threshold: categories that appear in < 0.01 % of rows  →  "other"
THRESHOLD = int(len(df) * 0.0001)          # ~42 rows for a 42 M-row file

for col in ["category_level1", "category_level2", "category_level3"]:
    freq = df[col].value_counts()
    rare = freq[freq < THRESHOLD].index
    df[col] = df[col].where(~df[col].isin(rare), other="other")

# Also collapse rare values in the raw category_code column
freq_code = df["category_code"].value_counts()
rare_code  = freq_code[freq_code < THRESHOLD].index
df["category_code"] = df["category_code"].where(
    ~df["category_code"].isin(rare_code), other="other"
)

# ── 7. Sort by user_id → event_time  (preserves session order naturally) ──────
df = df.sort_values(["user_id", "event_time"], kind="stable").reset_index(drop=True)

# ── 8. Final column order ─────────────────────────────────────────────────────
final_cols = [
    "user_id", "event_time", "event_type",
    "product_id", "category_id", "category_code",
    "category_level1", "category_level2", "category_level3",
    "brand", "price", "user_session",
]
df = df[final_cols]

print(f"\nFinal shape : {df.shape}")
print(df.dtypes)
print(df.head(10))

# ── 9. Export ─────────────────────────────────────────────────────────────────

# Parquet
df.to_parquet("event_level.parquet", index=False, engine="pyarrow")
print("Saved → event_level.parquet")

Raw shape: (42448764, 9)
event_time        object
event_type        object
product_id         int64
category_id        int64
category_code     object
brand             object
price            float64
user_id            int64
user_session      object
dtype: object
                event_time event_type  product_id          category_id  \
0  2019-10-01 00:00:00 UTC       view    44600062  2103807459595387724   
1  2019-10-01 00:00:00 UTC       view     3900821  2053013552326770905   
2  2019-10-01 00:00:01 UTC       view    17200506  2053013559792632471   

                         category_code     brand   price    user_id  \
0                                  NaN  shiseido   35.79  541312140   
1  appliances.environment.water_heater      aqua   33.20  554748717   
2           furniture.living_room.sofa       NaN  543.10  519107250   

                           user_session  
0  72d76fde-8bb3-4e00-8c23-a032dfed738c  
1  9333dfbd-b87a-4708-9857-6336556b0fcc  
2  566511c2-e2e3-422b-b695-c